In [0]:
# ============================================================
# CELL 1 — IMPORTS, PARAMETERS, AND CONFIGURATION
# Customers Bronze Ingestion
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from datetime import datetime, timezone

In [0]:
# ------------------------------------------------------------
# Storage paths — preserve the existing project layout
# ------------------------------------------------------------

STORAGE_ACCOUNT = "nestlesalesdatalake"
CONTAINER = "customer-sales-datalake"

BASE_PATH = (
    f"abfss://{CONTAINER}@"
    f"{STORAGE_ACCOUNT}.dfs.core.windows.net/"
)

LANDING_PATH = BASE_PATH + "landing/customers/"
BRONZE_PATH = BASE_PATH + "bronze/customers/"
MANIFEST_PATH = BASE_PATH + "metadata/file_manifest/"

In [0]:
# ------------------------------------------------------------
# Dataset configuration
# ------------------------------------------------------------

DATASET_NAME = "customers"
SOURCE_SYSTEM = "customer_api"

REQUIRED_SOURCE_COLUMNS = [
    "customer_id",
    "name",
    "city",
    "market"
]

BRONZE_COLUMNS = [
    "customer_id",
    "name",
    "city",
    "market",
    "ingestion_timestamp",
    "source_system",
    "source_file",
    "run_id",
    "batch_id"
]

In [0]:
# ------------------------------------------------------------
# Manifest run ID
# ------------------------------------------------------------

dbutils.widgets.text(
    "manifest_run_id",
    "fefa2f30-5d58-4043-8570-89b3e58146d0",
    "Manifest Run ID"
)

MANIFEST_RUN_ID = dbutils.widgets.get(
    "manifest_run_id"
).strip()

if not MANIFEST_RUN_ID:
    raise ValueError("Manifest Run ID cannot be empty.")

print("Dataset:", DATASET_NAME)
print("Manifest run:", MANIFEST_RUN_ID)
print("Landing path:", LANDING_PATH)
print("Bronze path:", BRONZE_PATH)
print("Manifest path:", MANIFEST_PATH)
print("Configuration initialized successfully.")


Dataset: customers
Manifest run: fefa2f30-5d58-4043-8570-89b3e58146d0
Landing path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/
Bronze path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/customers/
Manifest path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/file_manifest/
Configuration initialized successfully.


In [0]:

# ============================================================
# CELL 2 — LOAD AND VALIDATE CLAIMED CUSTOMER FILES
# ============================================================

# 1. Confirm the manifest exists
if not DeltaTable.isDeltaTable(spark, MANIFEST_PATH):
    raise RuntimeError(
        f"Manifest Delta table not found: {MANIFEST_PATH}"
    )

manifest_df = (
    spark.read
    .format("delta")
    .load(MANIFEST_PATH)
)

# 2. Validate required manifest columns
required_manifest_columns = {
    "dataset",
    "file_name",
    "file_path",
    "batch_id",
    "status",
    "run_id",
    "content_hash",
    "lease_expires_at"
}

missing_columns = (
    required_manifest_columns - set(manifest_df.columns)
)

if missing_columns:
    raise ValueError(
        f"Missing manifest columns: {sorted(missing_columns)}"
    )

# 3. Select only customer files owned by this run
claimed_files_df = (
    manifest_df
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("status") == "PROCESSING")
        & (F.col("run_id") == MANIFEST_RUN_ID)
    )
    .select(
        "dataset",
        "file_name",
        "file_path",
        "batch_id",
        "run_id",
        "content_hash",
        "lease_expires_at"
    )
)

# 4. Reject duplicate source identities
duplicate_files_df = (
    claimed_files_df
    .groupBy("dataset", "file_path")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_files_df.limit(1).count() > 0:
    display(duplicate_files_df)
    raise RuntimeError(
        "Duplicate claimed customer files found."
    )

claimed_count = claimed_files_df.count()

print("Customer files claimed by this run:", claimed_count)

if claimed_count != 3:
    display(claimed_files_df)
    raise RuntimeError(
        f"Expected 3 customer files for this run, "
        f"but found {claimed_count}. Stop and investigate."
    )

display(
    claimed_files_df.orderBy("file_name")
)


Customer files claimed by this run: 3


dataset,file_name,file_path,batch_id,run_id,content_hash,lease_expires_at
customers,customers_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json,17d146be-76e6-4412-81ac-c50156373bfa,fefa2f30-5d58-4043-8570-89b3e58146d0,5ef2c76d178ce3e2b35c8413a647bf9f5183d6404ffca94308d02a371d9765e0,2026-10-11T10:50:34.825Z
customers,customers_20261008.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=08/customers_20261008.json,212ac11d-5416-4c17-b936-a2b6c1c37c52,fefa2f30-5d58-4043-8570-89b3e58146d0,d82dc9b7776651362b54e203ec0734ae3a495534b7f093c9e2302e64033b5ef5,2026-10-11T10:50:34.825Z
customers,customers_20261009.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=09/customers_20261009.json,b1bd8ab0-8e65-4009-a8da-2ac857abc3d7,fefa2f30-5d58-4043-8570-89b3e58146d0,31246e22f0a2e56887ab66100a0c69832b3c77532d3a3197a1d80f6430a3b724,2026-10-11T10:50:34.825Z


In [0]:

# ============================================================
# CELL 3 — VERIFY CUSTOMER SOURCE HASHES AND LEASES
# Read-only validation
# ============================================================

from datetime import datetime, timezone
from pyspark.sql import functions as F

claimed_records = claimed_files_df.collect()

if len(claimed_records) != 3:
    raise RuntimeError("Expected exactly 3 claimed customer files.")

now_utc = datetime.now(timezone.utc)
validation_results = []

for row in claimed_records:
    file_path = row["file_path"]
    expected_hash = row["content_hash"]
    lease_expiry = row["lease_expires_at"]

    if not expected_hash:
        raise RuntimeError(
            f"Missing manifest content hash: {file_path}"
        )

    if lease_expiry is None:
        raise RuntimeError(
            f"Missing lease expiry: {file_path}"
        )

    lease_utc = (
        lease_expiry.replace(tzinfo=timezone.utc)
        if lease_expiry.tzinfo is None
        else lease_expiry.astimezone(timezone.utc)
    )

    if lease_utc <= now_utc:
        raise RuntimeError(
            f"Lease expired for {file_path}. Stop processing."
        )

    actual_hash = (
        spark.read.format("binaryFile")
        .load(file_path)
        .select(F.sha2(F.col("content"), 256).alias("hash"))
        .first()["hash"]
    )

    if actual_hash != expected_hash:
        raise RuntimeError(
            f"Source hash mismatch: {file_path}. "
            "Do not ingest this file."
        )

    validation_results.append(
        (row["file_name"], "VALID", actual_hash)
    )

validation_df = spark.createDataFrame(
    validation_results,
    ["file_name", "validation_status", "verified_hash"]
)

print("Customer files validated:", len(validation_results))
display(validation_df.orderBy("file_name"))


Customer files validated: 3


file_name,validation_status,verified_hash
customers_20261007.json,VALID,5ef2c76d178ce3e2b35c8413a647bf9f5183d6404ffca94308d02a371d9765e0
customers_20261008.json,VALID,d82dc9b7776651362b54e203ec0734ae3a495534b7f093c9e2302e64033b5ef5
customers_20261009.json,VALID,31246e22f0a2e56887ab66100a0c69832b3c77532d3a3197a1d80f6430a3b724


In [0]:

# ============================================================
# CELL 4 — READ AND VALIDATE CUSTOMER SOURCE RECORDS
# No Bronze writes in this cell
# ============================================================

from pyspark.sql import functions as F

customer_source_data = {}
customer_source_counts = {}

for row in claimed_records:
    file_name = row["file_name"]
    file_path = row["file_path"]

    print(f"Reading: {file_name}")

    source_df = (
        spark.read
        .option("multiLine", "true")
        .json(file_path)
    )

    missing_columns = [
        column
        for column in REQUIRED_SOURCE_COLUMNS
        if column not in source_df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{file_name} is missing required columns: "
            f"{missing_columns}. Available: {source_df.columns}"
        )

    # Count records and validate the required identifier.
    source_count = source_df.count()

    if source_count == 0:
        raise ValueError(f"{file_name} contains zero records.")

    if source_df.filter(
        F.col("customer_id").isNull()
    ).limit(1).count() > 0:
        raise ValueError(
            f"{file_name} contains a null customer_id."
        )

    customer_source_data[file_path] = source_df
    customer_source_counts[file_path] = source_count

    print(f"  Source records: {source_count}")

print("\nAll customer files passed source validation.")

summary_rows = [
    (row["file_name"], customer_source_counts[row["file_path"]])
    for row in claimed_records
]

source_summary_df = spark.createDataFrame(
    summary_rows,
    ["file_name", "source_record_count"]
)

display(source_summary_df.orderBy("file_name"))

print(
    "Total customer source records:",
    sum(customer_source_counts.values())
)


Reading: customers_20261007.json
  Source records: 25000
Reading: customers_20261008.json
  Source records: 25000
Reading: customers_20261009.json
  Source records: 25000

All customer files passed source validation.


file_name,source_record_count
customers_20261007.json,25000
customers_20261008.json,25000
customers_20261009.json,25000


Total customer source records: 75000


In [0]:

# ============================================================
# CELL 5 — PREPARE CUSTOMER BRONZE DATAFRAME
# No writes in this cell
# ============================================================

prepared_customer_dfs = []

for row in claimed_records:
    file_path = row["file_path"]
    batch_id = row["batch_id"]

    source_df = customer_source_data[file_path]

    bronze_df = (
        source_df
        .withColumn(
            "customer_id",
            F.col("customer_id").cast("int")
        )
        .withColumn("name", F.col("name").cast("string"))
        .withColumn("city", F.col("city").cast("string"))
        .withColumn("market", F.col("market").cast("string"))
        .withColumn(
            "ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "source_system",
            F.lit(SOURCE_SYSTEM)
        )
        .withColumn(
            "source_file",
            F.lit(file_path)
        )
        .withColumn(
            "run_id",
            F.lit(MANIFEST_RUN_ID)
        )
        .withColumn(
            "batch_id",
            F.lit(batch_id)
        )
        .select(*BRONZE_COLUMNS)
    )

    prepared_customer_dfs.append(bronze_df)

# Combine all three source files
prepared_customers_df = prepared_customer_dfs[0]

for next_df in prepared_customer_dfs[1:]:
    prepared_customers_df = prepared_customers_df.unionByName(next_df)

prepared_count = prepared_customers_df.count()

expected_count = sum(customer_source_counts.values())

if prepared_count != expected_count:
    raise RuntimeError(
        f"Record count mismatch: expected {expected_count}, "
        f"prepared {prepared_count}."
    )

print("Prepared Bronze records:", prepared_count)
print("Expected Bronze records:", expected_count)

prepared_customers_df.printSchema()

display(prepared_customers_df.limit(10))


Prepared Bronze records: 75000
Expected Bronze records: 75000
root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- market: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- source_system: string (nullable = false)
 |-- source_file: string (nullable = false)
 |-- run_id: string (nullable = false)
 |-- batch_id: string (nullable = false)



customer_id,name,city,market,ingestion_timestamp,source_system,source_file,run_id,batch_id
130001,Customer Oct7 Updated 1,Chennai,India,2026-10-11T10:02:54.811Z,customer_api,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json,fefa2f30-5d58-4043-8570-89b3e58146d0,17d146be-76e6-4412-81ac-c50156373bfa
130002,Customer Oct7 Updated 2,Kochi,India,2026-10-11T10:02:54.811Z,customer_api,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json,fefa2f30-5d58-4043-8570-89b3e58146d0,17d146be-76e6-4412-81ac-c50156373bfa
130003,Customer Oct7 Updated 3,Calgary,Canada,2026-10-11T10:02:54.811Z,customer_api,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json,fefa2f30-5d58-4043-8570-89b3e58146d0,17d146be-76e6-4412-81ac-c50156373bfa
130004,Customer Oct7 Updated 4,Bristol,UK,2026-10-11T10:02:54.811Z,customer_api,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json,fefa2f30-5d58-4043-8570-89b3e58146d0,17d146be-76e6-4412-81ac-c50156373bfa
130005,Customer Oct7 Updated 5,Perth,Australia,2026-10-11T10:02:54.811Z,customer_api,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json,fefa2f30-5d58-4043-8570-89b3e58146d0,17d146be-76e6-4412-81ac-c50156373bfa
130006,Customer Oct7 Updated 6,Hamburg,Germany,2026-10-11T10:02:54.811Z,customer_api,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json,fefa2f30-5d58-4043-8570-89b3e58146d0,17d146be-76e6-4412-81ac-c50156373bfa
160007,Customer Oct7 7,Ottawa,Canada,2026-10-11T10:02:54.811Z,customer_api,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json,fefa2f30-5d58-4043-8570-89b3e58146d0,17d146be-76e6-4412-81ac-c50156373bfa
160008,Customer Oct7 8,Cardiff,UK,2026-10-11T10:02:54.811Z,customer_api,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json,fefa2f30-5d58-4043-8570-89b3e58146d0,17d146be-76e6-4412-81ac-c50156373bfa
160009,Customer Oct7 9,Canberra,Australia,2026-10-11T10:02:54.811Z,customer_api,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json,fefa2f30-5d58-4043-8570-89b3e58146d0,17d146be-76e6-4412-81ac-c50156373bfa
160010,Customer Oct7 10,Hamburg,Germany,2026-10-11T10:02:54.811Z,customer_api,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json,fefa2f30-5d58-4043-8570-89b3e58146d0,17d146be-76e6-4412-81ac-c50156373bfa


In [0]:

# ============================================================
# CELL 6 — PRE-WRITE SAFETY CHECK
# No Bronze or manifest writes
# ============================================================

from pyspark.sql import functions as F

# A. Verify each file is still claimed by this run
latest_manifest_df = (
    spark.read
    .format("delta")
    .load(MANIFEST_PATH)
)

for row in claimed_records:
    matches = (
        latest_manifest_df
        .filter(
            (F.col("dataset") == DATASET_NAME)
            & (F.col("file_path") == row["file_path"])
            & (F.col("status") == "PROCESSING")
            & (F.col("run_id") == MANIFEST_RUN_ID)
        )
        .count()
    )

    if matches != 1:
        raise RuntimeError(
            f"Claim ownership check failed: {row['file_name']}"
        )

# B. Verify the Bronze target is empty for these source files.
# An absent Delta target is acceptable for a first load.
if DeltaTable.isDeltaTable(spark, BRONZE_PATH):
    existing_bronze_df = (
        spark.read
        .format("delta")
        .load(BRONZE_PATH)
    )

    if "source_file" not in existing_bronze_df.columns:
        raise RuntimeError(
            "Bronze table has no source_file column. Stop before writing."
        )

    source_paths = [row["file_path"] for row in claimed_records]

    existing_source_rows = (
        existing_bronze_df
        .filter(F.col("source_file").isin(source_paths))
        .count()
    )

    if existing_source_rows > 0:
        raise RuntimeError(
            f"Found {existing_source_rows} existing Bronze rows "
            "for these files. Stop to prevent duplicate ingestion."
        )

print("Claim ownership: VALID")
print("Existing Bronze rows for these files: 0")
print("Prepared records ready for controlled write:", prepared_count)
print("Pre-write safety checks passed.")


Claim ownership: VALID
Existing Bronze rows for these files: 0
Prepared records ready for controlled write: 75000
Pre-write safety checks passed.


In [0]:

# ============================================================
# CELL 7 — CONTROLLED BRONZE WRITE, FILE BY FILE
# ============================================================

from pyspark.sql import functions as F
from delta.tables import DeltaTable

successful_files = []
failed_files = []

for row in claimed_records:
    file_name = row["file_name"]
    file_path = row["file_path"]
    batch_id = row["batch_id"]
    expected_hash = row["content_hash"]

    print(f"\nProcessing: {file_name}")

    try:
        # 1. Confirm this run still owns the file
        current_row = (
            spark.read.format("delta").load(MANIFEST_PATH)
            .filter(
                (F.col("dataset") == DATASET_NAME)
                & (F.col("file_path") == file_path)
                & (F.col("status") == "PROCESSING")
                & (F.col("run_id") == MANIFEST_RUN_ID)
            )
            .select("content_hash", "lease_expires_at")
            .collect()
        )

        if len(current_row) != 1:
            raise RuntimeError("Manifest ownership check failed.")

        current_hash = current_row[0]["content_hash"]
        lease_expiry = current_row[0]["lease_expires_at"]

        if current_hash != expected_hash:
            raise RuntimeError("Manifest source hash changed.")

        if lease_expiry is None:
            raise RuntimeError("Manifest lease expiry is missing.")

        # 2. Recheck actual source content
        actual_hash = (
            spark.read.format("binaryFile")
            .load(file_path)
            .select(F.sha2(F.col("content"), 256).alias("hash"))
            .first()["hash"]
        )

        if actual_hash != expected_hash:
            raise RuntimeError("Source content hash mismatch.")

        # 3. Use the validated source DataFrame prepared earlier
        source_df = customer_source_data[file_path]
        source_count = customer_source_counts[file_path]

        bronze_file_df = (
            source_df
            .withColumn("customer_id", F.col("customer_id").cast("int"))
            .withColumn("name", F.col("name").cast("string"))
            .withColumn("city", F.col("city").cast("string"))
            .withColumn("market", F.col("market").cast("string"))
            .withColumn("ingestion_timestamp", F.current_timestamp())
            .withColumn("source_system", F.lit(SOURCE_SYSTEM))
            .withColumn("source_file", F.lit(file_path))
            .withColumn("run_id", F.lit(MANIFEST_RUN_ID))
            .withColumn("batch_id", F.lit(batch_id))
            .select(*BRONZE_COLUMNS)
        )

        # 4. Ensure this file has not already been written
        if DeltaTable.isDeltaTable(spark, BRONZE_PATH):
            existing = (
                spark.read.format("delta").load(BRONZE_PATH)
                .filter(F.col("source_file") == file_path)
                .count()
            )

            if existing != 0:
                raise RuntimeError(
                    f"Bronze already contains {existing} rows for this file. "
                    "Refusing to append duplicates."
                )

        # 5. Append this file
        (
            bronze_file_df.write
            .format("delta")
            .mode("append")
            .save(BRONZE_PATH)
        )

        # 6. Verify the persisted rows for this file
        persisted_count = (
            spark.read.format("delta").load(BRONZE_PATH)
            .filter(F.col("source_file") == file_path)
            .count()
        )

        if persisted_count != source_count:
            raise RuntimeError(
                f"Bronze verification failed: expected {source_count}, "
                f"found {persisted_count}."
            )

        # 7. Update manifest only after the Bronze write is verified
        update_manifest_status(
            dataset=DATASET_NAME,
            file_path=file_path,
            run_id=MANIFEST_RUN_ID,
            new_status="SUCCESS",
            source_record_count=source_count,
            written_record_count=persisted_count,
            rejected_record_count=0,
            processed_content_hash=actual_hash
        )

        successful_files.append(file_name)
        print(f"SUCCESS: {file_name} — {persisted_count} rows verified.")

    except Exception as exc:
        error_text = f"{type(exc).__name__}: {str(exc)}"[:4000]
        print(f"FAILED: {file_name} — {error_text}")

        try:
            # Don't overwrite a terminal state or another run's claim.
            current_status = (
                spark.read.format("delta").load(MANIFEST_PATH)
                .filter(
                    (F.col("dataset") == DATASET_NAME)
                    & (F.col("file_path") == file_path)
                )
                .select("status", "run_id")
                .collect()
            )

            if (
                len(current_status) == 1
                and current_status[0]["status"] == "PROCESSING"
                and current_status[0]["run_id"] == MANIFEST_RUN_ID
            ):
                update_manifest_status(
                    dataset=DATASET_NAME,
                    file_path=file_path,
                    run_id=MANIFEST_RUN_ID,
                    new_status="FAILED",
                    error_message=error_text,
                    error_type=type(exc).__name__
                )
        except Exception as status_exc:
            print("CRITICAL: Could not update failure status:",
                  str(status_exc)[:1000])

        failed_files.append((file_name, error_text))

print("\n========== CUSTOMER BRONZE RESULT ==========")
print("Successful files:", len(successful_files))
print("Failed files:", len(failed_files))
print("Successful file names:", successful_files)

if failed_files:
    display(
        spark.createDataFrame(
            failed_files, ["file_name", "error_message"]
        )
    )



Processing: customers_20261007.json
FAILED: customers_20261007.json — NameError: name 'update_manifest_status' is not defined
CRITICAL: Could not update failure status: name 'update_manifest_status' is not defined

Processing: customers_20261008.json
FAILED: customers_20261008.json — NameError: name 'update_manifest_status' is not defined
CRITICAL: Could not update failure status: name 'update_manifest_status' is not defined

Processing: customers_20261009.json
FAILED: customers_20261009.json — NameError: name 'update_manifest_status' is not defined
CRITICAL: Could not update failure status: name 'update_manifest_status' is not defined

========== CUSTOMER BRONZE RESULT ==========
Successful files: 0
Failed files: 3
Successful file names: []


file_name,error_message
customers_20261007.json,NameError: name 'update_manifest_status' is not defined
customers_20261008.json,NameError: name 'update_manifest_status' is not defined
customers_20261009.json,NameError: name 'update_manifest_status' is not defined


In [0]:

# ============================================================
# CELL 8 — INSPECT BRONZE WRITES AFTER THE ERROR
# Read-only: no data or status changes
# ============================================================

bronze_df = spark.read.format("delta").load(BRONZE_PATH)

for row in claimed_records:
    file_name = row["file_name"]
    file_path = row["file_path"]

    matching_df = bronze_df.filter(
        F.col("source_file") == file_path
    )

    row_count = matching_df.count()
    distinct_ids = matching_df.select("customer_id").distinct().count()

    print(
        f"{file_name}: rows={row_count}, "
        f"distinct_customer_ids={distinct_ids}"
    )

print("\nManifest status for this run:")

display(
    spark.read.format("delta").load(MANIFEST_PATH)
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("run_id") == MANIFEST_RUN_ID)
    )
    .select(
        "file_name",
        "status",
        "run_id",
        "error_message",
        "written_record_count",
        "processed_content_hash"
    )
    .orderBy("file_name")
)


customers_20261007.json: rows=25000, distinct_customer_ids=25000
customers_20261008.json: rows=25000, distinct_customer_ids=25000
customers_20261009.json: rows=25000, distinct_customer_ids=25000

Manifest status for this run:


file_name,status,run_id,error_message,written_record_count,processed_content_hash
customers_20261007.json,PROCESSING,fefa2f30-5d58-4043-8570-89b3e58146d0,null,null,null
customers_20261008.json,PROCESSING,fefa2f30-5d58-4043-8570-89b3e58146d0,null,null,null
customers_20261009.json,PROCESSING,fefa2f30-5d58-4043-8570-89b3e58146d0,null,null,null


In [0]:

# ============================================================
# CELL 9 — RECONCILE VERIFIED BRONZE WRITES WITH MANIFEST
# Does not write any additional Bronze records
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F

manifest_delta = DeltaTable.forPath(spark, MANIFEST_PATH)
bronze_df = spark.read.format("delta").load(BRONZE_PATH)

for row in claimed_records:
    file_name = row["file_name"]
    file_path = row["file_path"]
    expected_hash = row["content_hash"]

    print(f"Reconciling: {file_name}")

    # Verify that this run still owns the claim.
    owned = (
        spark.read.format("delta").load(MANIFEST_PATH)
        .filter(
            (F.col("dataset") == DATASET_NAME)
            & (F.col("file_path") == file_path)
            & (F.col("status") == "PROCESSING")
            & (F.col("run_id") == MANIFEST_RUN_ID)
            & (F.col("content_hash") == expected_hash)
        )
        .count()
    )

    if owned != 1:
        raise RuntimeError(
            f"Manifest ownership/hash check failed for {file_name}"
        )

    # Verify persisted Bronze count and required IDs.
    persisted = bronze_df.filter(
        F.col("source_file") == file_path
    )

    actual_count = persisted.count()
    distinct_ids = persisted.select("customer_id").distinct().count()
    null_ids = persisted.filter(
        F.col("customer_id").isNull()
    ).count()

    expected_count = customer_source_counts[file_path]

    if (
        actual_count != expected_count
        or distinct_ids != expected_count
        or null_ids != 0
    ):
        raise RuntimeError(
            f"Bronze verification failed for {file_name}: "
            f"rows={actual_count}, distinct IDs={distinct_ids}, "
            f"null IDs={null_ids}, expected={expected_count}"
        )

    # Update only the manifest row still owned by this run.
    (
        manifest_delta.alias("t")
        .update(
            condition=(
                (F.col("dataset") == F.lit(DATASET_NAME))
                & (F.col("file_path") == F.lit(file_path))
                & (F.col("status") == F.lit("PROCESSING"))
                & (F.col("run_id") == F.lit(MANIFEST_RUN_ID))
                & (F.col("content_hash") == F.lit(expected_hash))
            ),
            set={
                "status": F.lit("SUCCESS"),
                "processed_at": F.current_timestamp(),
                "status_updated_at": F.current_timestamp(),
                "last_success_at": F.current_timestamp(),
                "processed_content_hash": F.lit(expected_hash),
                "source_record_count": F.lit(expected_count).cast("long"),
                "written_record_count": F.lit(actual_count).cast("long"),
                "rejected_record_count": F.lit(0).cast("long"),
                "error_message": F.lit(None).cast("string"),
                "error_type": F.lit(None).cast("string"),
                "claimed_at": F.lit(None).cast("timestamp"),
                "lease_expires_at": F.lit(None).cast("timestamp")
            }
        )
    )

    print(
        f"Verified and reconciled: {file_name} — "
        f"{actual_count} Bronze rows"
    )

# Final status verification
final_status_df = (
    spark.read.format("delta").load(MANIFEST_PATH)
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("run_id") == MANIFEST_RUN_ID)
    )
    .select(
        "file_name",
        "status",
        "source_record_count",
        "written_record_count",
        "processed_content_hash"
    )
    .orderBy("file_name")
)

display(final_status_df)


Reconciling: customers_20261007.json
Verified and reconciled: customers_20261007.json — 25000 Bronze rows
Reconciling: customers_20261008.json
Verified and reconciled: customers_20261008.json — 25000 Bronze rows
Reconciling: customers_20261009.json
Verified and reconciled: customers_20261009.json — 25000 Bronze rows


file_name,status,source_record_count,written_record_count,processed_content_hash
customers_20261007.json,SUCCESS,25000,25000,5ef2c76d178ce3e2b35c8413a647bf9f5183d6404ffca94308d02a371d9765e0
customers_20261008.json,SUCCESS,25000,25000,d82dc9b7776651362b54e203ec0734ae3a495534b7f093c9e2302e64033b5ef5
customers_20261009.json,SUCCESS,25000,25000,31246e22f0a2e56887ab66100a0c69832b3c77532d3a3197a1d80f6430a3b724


In [0]:

# ============================================================
# CELL 10 — FINAL CUSTOMER BRONZE QUALITY CHECKS
# Read-only verification
# ============================================================

from pyspark.sql import functions as F

bronze_df = spark.read.format("delta").load(BRONZE_PATH)

# Overall record count
total_rows = bronze_df.count()

# Count records per source file
file_counts_df = (
    bronze_df
    .groupBy("source_file")
    .agg(
        F.count("*").alias("bronze_rows"),
        F.countDistinct("customer_id").alias("distinct_customer_ids"),
        F.sum(
            F.when(F.col("customer_id").isNull(), 1).otherwise(0)
        ).alias("null_customer_ids")
    )
    .orderBy("source_file")
)

display(file_counts_df)

print("Total Bronze customer rows:", total_rows)

if total_rows != 75000:
    raise RuntimeError(
        f"Expected 75000 Bronze rows, found {total_rows}."
    )

# Verify each file contains 25,000 rows and unique IDs.
bad_files_df = file_counts_df.filter(
    (F.col("bronze_rows") != 25000)
    | (F.col("distinct_customer_ids") != 25000)
    | (F.col("null_customer_ids") != 0)
)

if bad_files_df.limit(1).count() > 0:
    display(bad_files_df)
    raise RuntimeError("Customer Bronze quality checks failed.")

print("PASS: 75,000 records verified.")
print("PASS: Each source file has 25,000 distinct customer IDs.")


source_file,bronze_rows,distinct_customer_ids,null_customer_ids
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json,25000,25000,0
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=08/customers_20261008.json,25000,25000,0
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=09/customers_20261009.json,25000,25000,0


Total Bronze customer rows: 75000
PASS: 75,000 records verified.
PASS: Each source file has 25,000 distinct customer IDs.
